# Minimal H5 subset smoke test — samples as rows, proteins as columns

This notebook assumes the corrected pipeline files are already in the project.

It uses the real files in:

- `data/transcriptomics_filtered.joblib`
- `data/proteomics_matched.joblib`

and the orientation:

- transcriptomics: rows = samples, columns = genes
- proteomics: rows = samples, columns = proteins

It saves every generated per-protein `.h5` file in:

- `results_xgb/`

The default run is intentionally small: one protein, a small sample subset, and all normalization modes. Increase `N_PROTEINS` and `N_SAMPLES` after the smoke test passes.

In [2]:
from pathlib import Path
import os
import sys
import json
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import load

# Run this notebook from the project root.
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = (PROJECT_ROOT / "data").resolve()
RESULTS_DIR = (PROJECT_ROOT / "results_xgb").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "models").mkdir(parents=True, exist_ok=True)

# Small real-data subset settings. Override using environment variables if needed.
N_SAMPLES = int(os.environ.get("NB_N_SAMPLES", "12"))
N_PROTEINS_RAW = os.environ.get("NB_N_PROTEINS", "1")  # use "all" to run every protein
SELECT_K = int(os.environ.get("NB_SELECT_K", "50"))
SPLITS = int(os.environ.get("NB_SPLITS", "3"))
EARLY_STOPPING = int(os.environ.get("NB_EARLY_STOPPING", "5"))
MAX_PARAM_COMBOS = int(os.environ.get("NB_MAX_PARAM_COMBOS", "2"))
XGB_THREADS = int(os.environ.get("NB_XGB_THREADS", "2"))

# "tries everything" for the smoke test: all normalization modes.
NORMALIZATIONS = os.environ.get("NB_NORMALIZATIONS", "none,zscore,rank,qbin5,qbin10").split(",")
NORMALIZATIONS = [x.strip() for x in NORMALIZATIONS if x.strip()]

GRID_PRESET = os.environ.get("NB_GRID_PRESET", "tiny")
TUNING_STRATEGY = os.environ.get("NB_TUNING_STRATEGY", "once")
OUTPUT_SUFFIX_PREFIX = os.environ.get("NB_OUTPUT_SUFFIX_PREFIX", "notebook_minimal_h5_columns")
H5_MODEL = os.environ.get("NB_H5_MODEL", "best_loo")
H5_COMPRESSION_LEVEL = int(os.environ.get("NB_H5_COMPRESSION_LEVEL", "9"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("NORMALIZATIONS:", NORMALIZATIONS)
print("N_SAMPLES:", N_SAMPLES, "N_PROTEINS:", N_PROTEINS_RAW)
print("SELECT_K:", SELECT_K, "MAX_PARAM_COMBOS:", MAX_PARAM_COMBOS)

PROJECT_ROOT: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb
DATA_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data
RESULTS_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\results_xgb
NORMALIZATIONS: ['none', 'zscore', 'rank', 'qbin5', 'qbin10']
N_SAMPLES: 12 N_PROTEINS: 1
SELECT_K: 50 MAX_PARAM_COMBOS: 2


In [3]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

pipeline_py = first_existing([
    PROJECT_ROOT / "pipeline.py",
    PROJECT_ROOT / "src" / "pipeline.py",
])

pipeline_functions_py = first_existing([
    PROJECT_ROOT / "pipeline_functions.py",
    PROJECT_ROOT / "src" / "pipeline_functions.py",
])

run_sh = first_existing([
    PROJECT_ROOT / "run_xgboost.sh",
    PROJECT_ROOT / "run_xgboost_updated_cluster.sh",
])

X_path = DATA_DIR / "transcriptomics_filtered.joblib"
Y_path = DATA_DIR / "proteomics_matched.joblib"

required = {
    "pipeline.py": pipeline_py,
    "pipeline_functions.py": pipeline_functions_py,
    "transcriptomics_filtered.joblib": X_path if X_path.exists() else None,
    "proteomics_matched.joblib": Y_path if Y_path.exists() else None,
}

for name, path in required.items():
    print(f"{name:34s}: {path}")

missing = [name for name, path in required.items() if path is None]
if missing:
    raise FileNotFoundError("Missing required file(s): " + ", ".join(missing))

SRC_DIR = pipeline_py.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# pipeline.py reads inputs from DATA_DIR, while --metrics_dir writes outputs to RESULTS_DIR.
os.environ["DATA_DIR"] = str(DATA_DIR)
print("\nUsing SRC_DIR:", SRC_DIR)
print("Using DATA_DIR:", os.environ["DATA_DIR"])

pipeline.py                       : C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src\pipeline.py
pipeline_functions.py             : C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src\pipeline_functions.py
transcriptomics_filtered.joblib   : C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data\transcriptomics_filtered.joblib
proteomics_matched.joblib         : C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data\proteomics_matched.joblib

Using SRC_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src
Using DATA_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data


In [4]:
from pipeline_functions import load_minimal_protein_h5, predict_with_artifact

print("Imported H5 loader and artifact predictor from corrected pipeline_functions.py")

Imported H5 loader and artifact predictor from corrected pipeline_functions.py


In [5]:
# Load real data and detect orientation.
X = load(X_path)
Y = load(Y_path)

print("Transcriptomics shape:", X.shape)
print("Proteomics shape:", Y.shape)
print("Transcriptomics index examples:", list(map(str, X.index[:3])))
print("Proteomics row/sample examples:", list(map(str, Y.index[:3])))
print("Proteomics column/protein examples:", list(map(str, Y.columns[:3])))

sample_overlap_rows = len(pd.Index(X.index).intersection(pd.Index(Y.index)))
sample_overlap_cols = len(pd.Index(X.index).intersection(pd.Index(Y.columns)))

if sample_overlap_rows >= max(3, sample_overlap_cols):
    PROTEOMICS_ORIENTATION = "samples_as_rows__proteins_as_columns"
    protein_names = list(map(str, Y.columns))
    sample_axis = "index"
elif sample_overlap_cols >= 3:
    PROTEOMICS_ORIENTATION = "proteins_as_rows__samples_as_columns"
    protein_names = list(map(str, Y.index))
    sample_axis = "columns"
else:
    raise ValueError(
        "Could not detect proteomics orientation. Expected either Y.index or Y.columns to overlap X.index."
    )

print("Detected proteomics orientation:", PROTEOMICS_ORIENTATION)
print("Sample overlap using Y rows/index:", sample_overlap_rows)
print("Sample overlap using Y columns:", sample_overlap_cols)
print("Number of detected proteins:", len(protein_names))
print("First detected proteins:", protein_names[:5])

Transcriptomics shape: (369, 22614)
Proteomics shape: (369, 5113)
Transcriptomics index examples: ['MDAMB468_BREAST', 'SH4_SKIN', 'AU565_BREAST']
Proteomics row/sample examples: ['MDAMB468_BREAST', 'SH4_SKIN', 'AU565_BREAST']
Proteomics column/protein examples: ['A2M', 'AAAS', 'AACS']
Detected proteomics orientation: samples_as_rows__proteins_as_columns
Sample overlap using Y rows/index: 369
Sample overlap using Y columns: 0
Number of detected proteins: 5113
First detected proteins: ['A2M', 'AAAS', 'AACS', 'AAGAB', 'AAK1']


In [6]:
# Choose protein columns with enough non-missing values on the overlapping samples.
min_needed = max(N_SAMPLES, SPLITS + 1, 3)

if PROTEOMICS_ORIENTATION == "samples_as_rows__proteins_as_columns":
    overlapping_samples = pd.Index(X.index).intersection(pd.Index(Y.index))
    valid_proteins = []
    for protein in Y.columns:
        n_valid = int(Y.loc[overlapping_samples, protein].notna().sum())
        if n_valid >= min_needed:
            valid_proteins.append(str(protein))
else:
    overlapping_samples = pd.Index(X.index).intersection(pd.Index(Y.columns))
    valid_proteins = []
    for protein in Y.index:
        n_valid = int(Y.loc[protein, overlapping_samples].notna().sum())
        if n_valid >= min_needed:
            valid_proteins.append(str(protein))

if not valid_proteins:
    raise ValueError(f"No proteins found with at least {min_needed} overlapping non-missing samples.")

if N_PROTEINS_RAW.lower() == "all":
    selected_proteins = valid_proteins
else:
    selected_proteins = valid_proteins[: int(N_PROTEINS_RAW)]

print("Minimum non-missing samples required per protein:", min_needed)
print("Valid proteins:", len(valid_proteins))
print("Selected proteins:", selected_proteins[:10], "..." if len(selected_proteins) > 10 else "")

Minimum non-missing samples required per protein: 12
Valid proteins: 5113
Selected proteins: ['A2M'] 


In [6]:
def safe_component(value):
    return str(value).replace("/", "_").replace("\\", "_")

# Run pipeline.py in minimal_h5 mode and save all generated files into results_xgb/.
run_records = []

for protein in selected_proteins:
    safe_protein = safe_component(protein)
    for norm in NORMALIZATIONS:
        suffix = f"{OUTPUT_SUFFIX_PREFIX}_{norm}_{GRID_PRESET}_{TUNING_STRATEGY}_k{SELECT_K}_n{N_SAMPLES}"
        expected_h5 = RESULTS_DIR / f"{safe_protein}_{suffix}.h5"

        cmd = [
            sys.executable,
            str(pipeline_py),
            "--protein", protein,
            "--splits", str(SPLITS),
            "--select_k", str(SELECT_K),
            "--early_stopping", str(EARLY_STOPPING),
            "--n_samples", str(N_SAMPLES),
            "--max_param_combos", str(MAX_PARAM_COMBOS),
            "--normalization", norm,
            "--grid_preset", GRID_PRESET,
            "--tuning_strategy", TUNING_STRATEGY,
            "--xgb_threads", str(XGB_THREADS),
            "--minimal_save",
            "--output_mode", "minimal_h5",
            "--h5_compression_level", str(H5_COMPRESSION_LEVEL),
            "--h5_model", H5_MODEL,
            "--metrics_dir", str(RESULTS_DIR),
            "--model_dir", str(RESULTS_DIR / "models"),
            "--output_suffix", suffix,
        ]

        print("\n" + "=" * 100)
        print("Protein:", protein)
        print("Normalization:", norm)
        print("Expected H5:", expected_h5)
        print("Command:", " ".join(map(str, cmd)))

        completed = subprocess.run(
            cmd,
            cwd=str(SRC_DIR),
            env={**os.environ, "DATA_DIR": str(DATA_DIR)},
            text=True,
            capture_output=True,
        )

        print("Return code:", completed.returncode)
        print("--- STDOUT tail ---")
        print("\n".join(completed.stdout.splitlines()[-50:]))
        if completed.stderr.strip():
            print("--- STDERR tail ---")
            print("\n".join(completed.stderr.splitlines()[-50:]))

        exists = expected_h5.exists()
        run_records.append({
            "protein": protein,
            "normalization": norm,
            "returncode": completed.returncode,
            "h5_path": str(expected_h5),
            "exists": exists,
            "size_bytes": expected_h5.stat().st_size if exists else np.nan,
        })

        if completed.returncode != 0:
            raise RuntimeError(f"pipeline.py failed for protein={protein}, normalization={norm}.")
        if not exists:
            raise FileNotFoundError(f"Expected H5 file was not created: {expected_h5}")

runs_df = pd.DataFrame(run_records)
display(runs_df)


Protein: A2M
Normalization: none
Expected H5: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\results_xgb\A2M_notebook_minimal_h5_columns_none_tiny_once_k50_n12.h5
Command: c:\Users\hp\anaconda3\python.exe C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src\pipeline.py --protein A2M --splits 3 --select_k 50 --early_stopping 5 --n_samples 12 --max_param_combos 2 --normalization none --grid_preset tiny --tuning_strategy once --xgb_threads 2 --minimal_save --output_mode minimal_h5 --h5_compression_level 9 --h5_model best_loo --metrics_dir C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\results_xgb --model_dir C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\results_xgb\models --output_suffix notebook_minimal_h5_columns_none_tiny_once_k50_n12
Return code: 0
--- STDOUT tail ---
Loading data...
DATA_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data
METRICS

,protein,normalization,returncode,h5_path,exists,size_bytes
0,A2M,none,0,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,True,39280
1,A2M,zscore,0,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,True,39804
2,A2M,rank,0,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,True,41116
3,A2M,qbin5,0,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,True,40188
4,A2M,qbin10,0,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,True,41070


In [7]:
# Load and display one row per saved protein file: metrics + best model details.
loaded_h5 = {}
summary_rows = []

for rec in run_records:
    h5_path = rec["h5_path"]
    obj = load_minimal_protein_h5(h5_path)
    loaded_h5[(rec["protein"], rec["normalization"])] = obj

    metrics = obj["metrics"]
    model_artifact = obj["model_artifact"]
    best_summary = model_artifact.get("best_loo_model_summary", {}) or {}

    summary_rows.append({
        "protein": metrics.get("protein", rec["protein"]),
        "normalization": rec["normalization"],
        "pearson_corr": metrics.get("pearson_corr"),
        "mse": metrics.get("mse"),
        "rmse": metrics.get("rmse"),
        "cosine_similarity": metrics.get("vector_cosine_similarity_all_loo"),
        "n_valid_predictions": metrics.get("n_valid_predictions"),
        "saved_model_label": metrics.get("h5_model_label"),
        "artifact_kind": model_artifact.get("artifact_kind"),
        "best_model_held_out_sample": best_summary.get("held_out_sample"),
        "best_model_squared_error": best_summary.get("squared_error"),
        "selected_feature_count": len(model_artifact.get("selected_features", [])),
        "model_params": model_artifact.get("params"),
        "model_object": model_artifact.get("model"),
        "h5_path": rec["h5_path"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["protein", "pearson_corr"], ascending=[True, False], na_position="last")
display(summary_df)

,protein,normalization,pearson_corr,mse,rmse,cosine_similarity,n_valid_predictions,saved_model_label,artifact_kind,best_model_held_out_sample,best_model_squared_error,selected_feature_count,model_params,model_object,h5_path
0,A2M,none,0.123918,1.626113,1.275191,0.219167,12,best_loo_model_artifact,best_loo_candidate_model,SKHEP1_LIVER,0.018437,50,"{'max_depth': 4, 'min_child_weight': 3, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...
1,A2M,zscore,0.123918,1.626113,1.275191,0.219167,12,best_loo_model_artifact,best_loo_candidate_model,SKHEP1_LIVER,0.018437,50,"{'max_depth': 4, 'min_child_weight': 3, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...
2,A2M,rank,0.123918,1.626113,1.275191,0.219167,12,best_loo_model_artifact,best_loo_candidate_model,SKHEP1_LIVER,0.018437,50,"{'max_depth': 4, 'min_child_weight': 3, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...
4,A2M,qbin10,-0.041417,1.765587,1.328754,0.110783,12,best_loo_model_artifact,best_loo_candidate_model,SKHEP1_LIVER,0.088556,50,"{'max_depth': 4, 'min_child_weight': 3, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...
3,A2M,qbin5,-0.856904,2.546920,1.595907,-0.293254,12,best_loo_model_artifact,best_loo_candidate_model,CAL51_BREAST,0.169991,50,"{'max_depth': 3, 'min_child_weight': 1, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...


In [8]:
# Pick the best saved file by Pearson and print the saved protein row with metrics + model.
best_idx = summary_df["pearson_corr"].astype(float).idxmax() if summary_df["pearson_corr"].notna().any() else summary_df.index[0]
best_row = summary_df.loc[[best_idx]]
best_protein = best_row.iloc[0]["protein"]
best_norm = best_row.iloc[0]["normalization"]
best_obj = loaded_h5[(best_protein, best_norm)]
best_model = best_obj["model_artifact"]

print("Best saved H5 by Pearson:")
print(best_row.iloc[0]["h5_path"])
print("\nBest model params:")
print(json.dumps(best_model.get("params", {}), indent=2, sort_keys=True))
print("\nBest LOO model summary:")
print(json.dumps(best_model.get("best_loo_model_summary", {}), indent=2, sort_keys=True))

# This is the requested saved protein row: metrics + model artifact details.
display(best_row)

Best saved H5 by Pearson:
C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\results_xgb\A2M_notebook_minimal_h5_columns_none_tiny_once_k50_n12.h5

Best model params:
{
  "colsample_bytree": 0.8,
  "device": "cpu",
  "gamma": 0.0,
  "learning_rate": 0.05,
  "max_bin": 256,
  "max_depth": 4,
  "min_child_weight": 3,
  "n_estimators": 25,
  "reg_alpha": 0.1,
  "reg_lambda": 1.0,
  "subsample": 0.9,
  "tree_method": "hist"
}

Best LOO model summary:
{
  "absolute_error": 0.13578144659214292,
  "held_out_sample": "SKHEP1_LIVER",
  "params": {
    "_cv_median_best_iteration": 3,
    "_cv_mse": 1.3357915182908375,
    "colsample_bytree": 0.8,
    "device": "cpu",
    "gamma": 0.0,
    "learning_rate": 0.05,
    "max_bin": 256,
    "max_depth": 4,
    "min_child_weight": 3,
    "n_estimators": 25,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "subsample": 0.9,
    "tree_method": "hist"
  },
  "predicted": -0.28350669145584106,
  "selected_feature_count": 50,
  "squar

,protein,normalization,pearson_corr,mse,rmse,cosine_similarity,n_valid_predictions,saved_model_label,artifact_kind,best_model_held_out_sample,best_model_squared_error,selected_feature_count,model_params,model_object,h5_path
0,A2M,none,0.123918,1.626113,1.275191,0.219167,12,best_loo_model_artifact,best_loo_candidate_model,SKHEP1_LIVER,0.018437,50,"{'max_depth': 4, 'min_child_weight': 3, 'learn...","XGBRegressor(base_score=None, booster=None, ca...",C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...


In [9]:
# Display all LOO true/predicted values from the best saved H5.
predictions_df = best_obj["predictions"].copy()
predictions_df["residual"] = predictions_df["true"] - predictions_df["predicted"]
display(predictions_df)

,protein,LOO sample,true,predicted,squared_error,absolute_error,residual
0,A2M,MDAMB468_BREAST,-0.345562,-0.111211,0.054920,0.234351,-0.234351
1,A2M,SH4_SKIN,1.642186,0.152462,2.219278,1.489724,1.489724
2,A2M,AU565_BREAST,-1.129114,1.216512,5.501963,2.345626,-2.345626
3,A2M,KMRC1_KIDNEY,1.493333,1.041700,0.203973,0.451634,0.451634
4,A2M,CAL51_BREAST,0.274652,-0.450489,0.525830,0.725141,0.725141
5,A2M,RPMI7951_SKIN,-0.615667,0.910993,2.330691,1.526660,-1.526660
6,A2M,RERFLCMS_LUNG,-0.957249,0.063630,1.042193,1.020878,-1.020878
7,A2M,IGR37_SKIN,-0.174494,0.675707,0.722842,0.850201,-0.850201
8,A2M,VMRCRCW_KIDNEY,-0.987932,1.037741,4.103352,2.025673,-2.025673
9,A2M,HEP3B217_LIVER,1.513325,0.972004,0.293029,0.541322,0.541322


In [10]:
# Test that the saved model artifact can predict on real transcriptomics rows.
X_for_prediction = X.iloc[: min(5, len(X))].copy()
artifact_predictions = predict_with_artifact(best_model, X_for_prediction)

pred_check_df = artifact_predictions.reset_index()
pred_check_df.columns = ["sample", "saved_model_prediction"]
display(pred_check_df)

,sample,saved_model_prediction
0,MDAMB468_BREAST,-0.091192
1,SH4_SKIN,1.194543
2,AU565_BREAST,-0.418551
3,KMRC1_KIDNEY,1.194543
4,CAL51_BREAST,0.224042


In [11]:
# List all H5 protein files currently in results_xgb/.
all_h5 = sorted(RESULTS_DIR.glob("*.h5"))
files_df = pd.DataFrame({
    "file": [p.name for p in all_h5],
    "path": [str(p) for p in all_h5],
    "size_bytes": [p.stat().st_size for p in all_h5],
})
display(files_df)

,file,path,size_bytes
0,A2M_notebook_minimal_h5_columns_none_tiny_once...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,39280
1,A2M_notebook_minimal_h5_columns_qbin10_tiny_on...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,41070
2,A2M_notebook_minimal_h5_columns_qbin5_tiny_onc...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,40188
3,A2M_notebook_minimal_h5_columns_rank_tiny_once...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,41116
4,A2M_notebook_minimal_h5_columns_zscore_tiny_on...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,39804


In [7]:
# Optional shell-script orientation check. This does not modify or submit the shell script.
# With your data orientation, run_xgboost.sh must read protein names from Y.columns, not Y.index.
if run_sh is None:
    print("No run_xgboost.sh found.")
else:
    text = run_sh.read_text(encoding="utf-8")
    print("Shell script:", run_sh)
    if "proteins = list(getattr(Y, \"index\", []))" in text or "proteins = list(Y.index)" in text:
        print("WARNING: this shell script appears to build the protein list from Y.index.")
        print("For this dataset, proteomics rows are samples and columns are proteins, so the shell script should use Y.columns.")
    else:
        print("No obvious Y.index protein-list line found, or the shell script may already be corrected.")

    for i, line in enumerate(text.splitlines(), start=1):
        if any(term in line for term in ["OUTPUT_MODE=", "H5_COMPRESSION_LEVEL=", "H5_MODEL=", "--output_mode", "--metrics_dir", "results_xgb", "Y.index", "Y.columns"]):
            print(f"{i:04d}: {line}")

Shell script: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\run_xgboost.sh
For this dataset, proteomics rows are samples and columns are proteins, so the shell script should use Y.columns.
0009: RESULTS_DIR=${RESULTS_DIR:-$PWD/results_xgb}
0048: # Output mode. Default is one compressed H5 per protein in results_xgb/.
0049: OUTPUT_MODE=${OUTPUT_MODE:-minimal_h5}  # minimal_h5 | full_joblib
0050: H5_COMPRESSION_LEVEL=${H5_COMPRESSION_LEVEL:-9}
0051: H5_MODEL=${H5_MODEL:-best_loo}          # best_loo | final
0120:   echo "ERROR: OUTPUT_MODE=minimal_h5 requires SAVE_MODELS=1 because the H5 stores the model artifact."
0153: # so auto detection uses Y.columns as the protein list.
0208:         f"Sample overlap with Y.index={index_overlap}, Y.columns={column_overlap}. "
0286: echo "Pipeline: USE_GPU=${USE_GPU}, SELECT_K=${SELECT_K}, NORMALIZATIONS=${NORMALIZATIONS}, GRID_PRESET=${GRID_PRESET}, TUNING_STRATEGY=${TUNING_STRATEGY}, OUTPUT_MODE=${OUTPUT_MODE}, H5_MODEL